## Sprint 3 — DB Integration & CLI Wiring (`db_manager`, `risk_queries.py`, `cli.py`)
 This is the last piece
of `Sprint3_Code_Implementation.docx` — the glue that lets the CLI read/write
the Sprint 3 risk tables and trigger training from the terminal menu (options
17-19), rather than analysis that benefits from being its own notebook.

### Why this one's different from every notebook before it

Everything up to now (EDA, feature engineering, model training, clustering,
charts, GRU) works directly off CSVs and saved model files — no database
required. `risk_queries.py` and `db_manager.init_risk_db()` are different: they
**only make sense against a live MySQL connection**, and this sandbox doesn't
have a MySQL server running.

So this notebook uses **SQLite as a stand-in** to prove the query logic and
round-trip actually work, with every MySQL-specific piece called out explicitly
so nothing here is copy-paste-able as-is into your MySQL project without the
noted substitution reversed:

| MySQL (production, `db/schema_risk.sql`) | SQLite (this notebook's stand-in) |
|---|---|
| `INT AUTO_INCREMENT PRIMARY KEY` | `INTEGER PRIMARY KEY AUTOINCREMENT` |
| `%s` placeholders (`mysql-connector`) | `?` placeholders (Python's built-in `sqlite3`) |
| `mysql.connector.errors.IntegrityError` | `sqlite3.IntegrityError` |
| `ENGINE=InnoDB` | (no equivalent needed) |

Everything else — the actual SQL logic in `get_latest_predictions()`'s
subquery, the `INSERT`/`ON DUPLICATE`-style dedup pattern, the handler flow in
`cli.py` — is unchanged from the docx.

### What this notebook does, in order
1. Explain `db/schema_risk.sql`'s three tables
2. Stand up the schema in SQLite (`init_risk_db()` equivalent)
3. Load it with the UCI/OULAD data + risk predictions already built in earlier
   notebooks (`data_migration.py` + `model_baseline.predict_risk()`'s output)
4. Define and run every `risk_queries.py` function against it for real
5. Walk through `cli.py`'s three new menu handlers (17, 18, 19), simulated
   non-interactively since a notebook can't take live terminal `input()`
6. Summary — the full Sprint 3 pipeline, module by module

In [1]:
import os
import sqlite3
import json
import numpy as np
import pandas as pd
import joblib

PROCESSED_DIR = os.path.join("..", "data", "processed")
MODELS_DIR = os.path.join("..", "models")

conn = sqlite3.connect(":memory:")   # stand-in for a real MySQL connection
print("SQLite connection open (in-memory -- stand-in for db_manager.get_connection())")

SQLite connection open (in-memory -- stand-in for db_manager.get_connection())


## 1. `db_manager.init_risk_db()` — create the Sprint 3 tables

The three tables from `db/schema_risk.sql`, translated to SQLite syntax per
the table above. Column meanings, unchanged from the schema:

- **`uci_students`** — one row per UCI student (source_row_id keeps it traceable
  back to the original CSV row); `g3` is kept so `at_risk` can be re-derived if
  the pass threshold ever changes, without re-migrating.
- **`oulad_engagement`** — one row per (student, week) — long format, matching
  what `feature_engineering_oulad.ipynb` streamed from `studentVle.csv`.
- **`risk_predictions`** — model output. `source_type` (`'uci' | 'oulad'`)
  keeps the two populations distinguishable in one shared predictions table,
  consistent with "never joined, always labeled" from the architecture doc.

In [2]:
def init_risk_db(conn):
    """SQLite translation of db_manager.init_risk_db(). Idempotent: CREATE TABLE
    IF NOT EXISTS, same as the MySQL original -- safe to call every CLI startup."""
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS uci_students (
            uci_student_id  INTEGER PRIMARY KEY AUTOINCREMENT,
            source_row_id   TEXT UNIQUE,
            school TEXT, sex TEXT, age INTEGER, address TEXT, famsize TEXT,
            medu INTEGER, fedu INTEGER, studytime INTEGER, failures INTEGER,
            schoolsup TEXT, famsup TEXT, higher TEXT, internet TEXT, romantic TEXT,
            famrel INTEGER, freetime INTEGER, goout INTEGER, health INTEGER,
            absences INTEGER, g1 REAL, g2 REAL, g3 REAL
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS oulad_engagement (
            engagement_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            oulad_student_id  TEXT NOT NULL,
            week_number       INTEGER,
            total_clicks      INTEGER,
            days_active       INTEGER,
            module_presentation TEXT,
            final_result      TEXT,
            UNIQUE(oulad_student_id, week_number)
        )
    """)
    cur.execute("CREATE INDEX IF NOT EXISTS idx_oulad_student ON oulad_engagement(oulad_student_id)")

    cur.execute("""
        CREATE TABLE IF NOT EXISTS risk_predictions (
            prediction_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            student_ref     TEXT NOT NULL,
            source_type     TEXT NOT NULL,
            risk_score      REAL NOT NULL,
            predicted_label TEXT,
            model_version   TEXT,
            scored_at       DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    """)
    cur.execute("CREATE INDEX IF NOT EXISTS idx_risk_predictions_ref ON risk_predictions(student_ref, source_type)")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_risk_predictions_scored_at ON risk_predictions(scored_at)")

    conn.commit()
    cur.close()

init_risk_db(conn)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables created:")
print(tables)

Tables created:
               name
0      uci_students
1   sqlite_sequence
2  oulad_engagement
3  risk_predictions


## 2. Load real data into the tables (stand-in for `data_migration.py`)

`data_migration.py` isn't its own notebook (it's straightforward CSV-to-SQL
loading, already effectively demonstrated by `feature_engineering_uci.ipynb`'s
Section 1, which reproduces exactly what lands in `uci_students`). Here we
actually write that data into the SQLite tables, so the query functions below
have something real to query.

In [3]:
# uci_students -- reload raw CSVs the same way feature_engineering_uci.ipynb did
def load_uci_raw(filename):
    path = os.path.join("..", "data", "raw", "uci_student_performance", filename)
    df = pd.read_csv(path, sep=";")
    df.columns = [c.lower() for c in df.columns]
    stem = filename.replace(".csv", "")
    df["source_row_id"] = stem + "_" + df.index.astype(str)
    return df

mat = load_uci_raw("student-mat.csv")
por = load_uci_raw("student-por.csv")
uci_raw = pd.concat([mat, por], ignore_index=True)

schema_cols = [
    "source_row_id", "school", "sex", "age", "address", "famsize", "medu", "fedu",
    "studytime", "failures", "schoolsup", "famsup", "higher", "internet", "romantic",
    "famrel", "freetime", "goout", "health", "absences", "g1", "g2", "g3",
]
uci_raw[schema_cols].to_sql("uci_students", conn, if_exists="append", index=False)
print(f"Loaded {len(uci_raw)} rows into uci_students")

Loaded 1044 rows into uci_students


In [4]:
# oulad_engagement -- reuse the weekly aggregation built in feature_engineering_oulad.ipynb's
# approach, but keep it in long (student, week) format to match the schema exactly
def read_oulad(name, **kwargs):
    return pd.read_csv(os.path.join("..", "data", "raw", "oulad", name), na_values="?", **kwargs)

student_info = read_oulad("studentInfo.csv")
outcome_rank = {"Distinction": 0, "Pass": 1, "Fail": 2, "Withdrawn": 3}
info_dedup = (
    student_info.assign(_rank=student_info["final_result"].map(outcome_rank))
    .sort_values("_rank", ascending=False)
    .drop_duplicates(subset="id_student", keep="first")
)
outcome_lookup = info_dedup.set_index("id_student")[["code_module", "final_result"]]

vle_path = os.path.join("..", "data", "raw", "oulad", "studentVle.csv")
CHUNK = 1_000_000
weekly_clicks, weekly_days = {}, {}

for chunk in pd.read_csv(vle_path, na_values="?", chunksize=CHUNK,
                          usecols=["id_student", "date", "sum_click"]):
    chunk["week"] = (chunk["date"].clip(lower=0) // 7).astype(int)
    g_clicks = chunk.groupby(["id_student", "week"])["sum_click"].sum()
    for (sid, wk), val in g_clicks.items():
        weekly_clicks[(sid, wk)] = weekly_clicks.get((sid, wk), 0) + val
    g_days = chunk.groupby(["id_student", "week"])["date"].apply(lambda s: len(set(s.unique())))
    for (sid, wk), n in g_days.items():
        weekly_days[(sid, wk)] = weekly_days.get((sid, wk), 0) + n

oulad_rows = []
for (sid, wk), clicks in weekly_clicks.items():
    info = outcome_lookup.loc[sid] if sid in outcome_lookup.index else None
    oulad_rows.append({
        "oulad_student_id": str(sid),
        "week_number": wk,
        "total_clicks": int(clicks),
        "days_active": int(weekly_days.get((sid, wk), 0)),
        "module_presentation": info["code_module"] if info is not None else None,
        "final_result": info["final_result"] if info is not None else None,
    })

oulad_engagement_df = pd.DataFrame(oulad_rows)
oulad_engagement_df.to_sql("oulad_engagement", conn, if_exists="append", index=False)
print(f"Loaded {len(oulad_engagement_df):,} rows into oulad_engagement")

Loaded 544,042 rows into oulad_engagement


In [5]:
# risk_predictions -- load the already-computed risk scores from the model
# training notebooks (this is what model_baseline.predict_risk() produced)
uci_risk = pd.read_csv(os.path.join(PROCESSED_DIR, "uci_risk_scores.csv"))
oulad_risk = pd.read_csv(os.path.join(PROCESSED_DIR, "oulad_risk_scores.csv"))

print(f"UCI risk predictions available to save: {len(uci_risk):,}")
print(f"OULAD risk predictions available to save: {len(oulad_risk):,}")
uci_risk.head(3)

UCI risk predictions available to save: 1,044
OULAD risk predictions available to save: 26,074


,risk_score,predicted_label,risk_level,actual_at_risk
0,0.9107,at_risk,High,1
1,0.9304,at_risk,High,1
2,0.7514,at_risk,High,0


## 3. `risk_queries.py` — defined and run for real

In [6]:
def get_uci_students(conn) -> pd.DataFrame:
    """Fetch all rows from uci_students. Returns empty DataFrame if table is empty."""
    return pd.read_sql("SELECT * FROM uci_students", conn)

def get_oulad_engagement(conn) -> pd.DataFrame:
    """Fetch all rows from oulad_engagement. Returns empty DataFrame if table is empty."""
    return pd.read_sql("SELECT * FROM oulad_engagement", conn)

df_uci_check = get_uci_students(conn)
df_oulad_check = get_oulad_engagement(conn)
print(f"get_uci_students(conn): {len(df_uci_check):,} rows")
print(f"get_oulad_engagement(conn): {len(df_oulad_check):,} rows")
df_uci_check.head(3)

get_uci_students(conn): 1,044 rows
get_oulad_engagement(conn): 544,042 rows


,uci_student_id,source_row_id,school,sex,age,address,famsize,medu,fedu,studytime,...,internet,romantic,famrel,freetime,goout,health,absences,g1,g2,g3
0,1,student-mat_0,GP,F,18,U,GT3,4,4,2,...,no,no,4,3,4,3,6,5.0,6.0,6.0
1,2,student-mat_1,GP,F,17,U,GT3,1,1,2,...,yes,no,5,3,3,3,4,5.0,5.0,6.0
2,3,student-mat_2,GP,F,15,U,LE3,1,1,2,...,yes,no,4,3,2,3,10,7.0,8.0,10.0


In [7]:
def save_predictions(conn, predictions_df: pd.DataFrame, source_type: str = "uci",
                      model_version: str = "xgboost_v1") -> dict:
    """
    Persist predict_risk() output to risk_predictions.
    predictions_df must contain: risk_score, predicted_label.
    Optionally: student_ref (falls back to row index).

    SQLite note: uses `?` placeholders and sqlite3.IntegrityError, vs. the
    production version's `%s` placeholders and mysql.connector's IntegrityError
    -- the INSERT logic and inserted/skipped bookkeeping are otherwise identical.
    """
    SQL = """
        INSERT INTO risk_predictions
            (student_ref, source_type, risk_score, predicted_label, model_version)
        VALUES (?, ?, ?, ?, ?)
    """
    inserted, skipped = 0, 0
    cur = conn.cursor()
    for idx, row in predictions_df.iterrows():
        student_ref = str(row.get("student_ref", idx))
        try:
            cur.execute(SQL, (
                student_ref, source_type,
                float(row["risk_score"]), str(row["predicted_label"]), model_version,
            ))
            inserted += 1
        except sqlite3.IntegrityError:
            skipped += 1
    conn.commit()
    cur.close()
    return {"inserted": inserted, "skipped": skipped}

uci_save_result = save_predictions(conn, uci_risk, source_type="uci", model_version="xgboost_uci_v1")
oulad_save_result = save_predictions(conn, oulad_risk, source_type="oulad", model_version="xgboost_oulad_v1")
print(f"UCI predictions saved:   {uci_save_result}")
print(f"OULAD predictions saved: {oulad_save_result}")

UCI predictions saved:   {'inserted': 1044, 'skipped': 0}
OULAD predictions saved: {'inserted': 26074, 'skipped': 0}


In [8]:
def get_latest_predictions(conn, threshold: float = None) -> pd.DataFrame:
    """
    Return the most recent risk_predictions row per (student_ref, source_type).
    The subquery finds MAX(scored_at) per student -- ensures we see each
    student's current risk score rather than historical re-scores.

    threshold: if provided, only return rows with risk_score >= threshold
               (e.g. threshold=0.5 returns only 'at_risk' students).

    Identical SQL logic to the MySQL original -- this query is portable ANSI SQL,
    no MySQL-specific syntax to translate.
    """
    sql = """
        SELECT rp.*
        FROM risk_predictions rp
        INNER JOIN (
            SELECT student_ref, source_type, MAX(scored_at) AS latest
            FROM risk_predictions
            GROUP BY student_ref, source_type
        ) latest_rows
            ON  rp.student_ref  = latest_rows.student_ref
            AND rp.source_type  = latest_rows.source_type
            AND rp.scored_at    = latest_rows.latest
    """
    if threshold is not None:
        sql += f" WHERE rp.risk_score >= {float(threshold)}"
    sql += " ORDER BY rp.risk_score DESC"
    return pd.read_sql(sql, conn)

at_risk_students = get_latest_predictions(conn, threshold=0.5)
print(f"Students with risk_score >= 0.5: {len(at_risk_students):,}")
print(f"  from UCI:   {(at_risk_students['source_type'] == 'uci').sum():,}")
print(f"  from OULAD: {(at_risk_students['source_type'] == 'oulad').sum():,}")
at_risk_students.sort_values("risk_score", ascending=False).head(10)

Students with risk_score >= 0.5: 11,567
  from UCI:   250
  from OULAD: 11,317


,prediction_id,student_ref,source_type,risk_score,predicted_label,model_version,scored_at
0,12730,11685,oulad,1.0000,at_risk,xgboost_oulad_v1,2026-08-12 19:36:07
1,13548,12503,oulad,1.0000,at_risk,xgboost_oulad_v1,2026-08-12 19:36:07
2,15629,14584,oulad,1.0000,at_risk,xgboost_oulad_v1,2026-08-12 19:36:07
3,20740,19695,oulad,1.0000,at_risk,xgboost_oulad_v1,2026-08-12 19:36:07
4,21271,20226,oulad,1.0000,at_risk,xgboost_oulad_v1,2026-08-12 19:36:07
5,23491,22446,oulad,1.0000,at_risk,xgboost_oulad_v1,2026-08-12 19:36:08
6,26833,25788,oulad,1.0000,at_risk,xgboost_oulad_v1,2026-08-12 19:36:08
7,8538,7493,oulad,1.0000,at_risk,xgboost_oulad_v1,2026-08-12 19:36:07
8,10667,9622,oulad,1.0000,at_risk,xgboost_oulad_v1,2026-08-12 19:36:07
9,11059,10014,oulad,0.9999,at_risk,xgboost_oulad_v1,2026-08-12 19:36:07


**Round trip confirmed:** predictions saved via `save_predictions()` come back
correctly through `get_latest_predictions()`, correctly split by `source_type`,
correctly sorted by risk descending. This is the exact query
`cli.py`'s option 19 (`handle_show_risk`) calls.

## 4. `cli.py` — walking through options 17, 18, 19

The CLI itself can't run inside a notebook (`input()` needs a live terminal,
`run()` is an infinite loop) — so instead of pretending to execute it, this
section calls the exact same underlying functions each handler calls, in the
same order, printing the same messages, so you can see precisely what typing
`17`, `18`, then `19` at the menu would produce.

In [9]:
print("=" * 60)
print("Simulating: user selects option 17 (Migrate Risk Data)")
print("=" * 60)
print()
print("  >>> db_manager.init_risk_db()   # already done in Section 1 above")
print()
print("Migrating UCI + OULAD data into MySQL...")
print(f"  uci_students: {{'inserted': {len(uci_raw)}, 'skipped': 0}}")
print(f"  oulad_engagement: {{'inserted': {len(oulad_engagement_df)}, 'skipped': 0}}")
print()
print("(This is exactly what happened in Section 2 above -- handle_migrate_risk()")
print(" just calls data_migration.migrate_all(conn), which wraps the same loads.)")

Simulating: user selects option 17 (Migrate Risk Data)

  >>> db_manager.init_risk_db()   # already done in Section 1 above

Migrating UCI + OULAD data into MySQL...
  uci_students: {'inserted': 1044, 'skipped': 0}
  oulad_engagement: {'inserted': 544042, 'skipped': 0}

(This is exactly what happened in Section 2 above -- handle_migrate_risk()
 just calls data_migration.migrate_all(conn), which wraps the same loads.)


In [10]:
print("=" * 60)
print("Simulating: user selects option 18 (Train Risk Model), then enters '2' for OULAD")
print("=" * 60)
print()

# handle_train_risk() calls feature_engineering.build_risk_feature_matrix(conn, source),
# then model_baseline.split_train_test / train_baseline_model / predict_risk.
# We already have all of that done -- reload the already-trained OULAD model and
# its saved test-set metrics rather than retraining, and print in the same shape
# handle_train_risk() would.
oulad_model = joblib.load(os.path.join(MODELS_DIR, "risk_model_oulad_v1.joblib"))
with open(os.path.join(MODELS_DIR, "risk_model_oulad_v1_summary.json")) as f:
    oulad_summary = json.load(f)

print("Training XGBoost classifier...")
print(f"\n  Precision (at-risk): {oulad_summary['test_precision_at_risk']:.3f}")
print(f"  Recall   (at-risk): {oulad_summary['test_recall_at_risk']:.3f}")
print(f"  F1 (at-risk):       {oulad_summary['test_f1_at_risk']:.3f}")
print(f"  ROC-AUC:            {oulad_summary['test_roc_auc']:.3f}")
print(f"\nModel saved to: {os.path.join(MODELS_DIR, 'risk_model_oulad_v1.joblib')}")
print("Predictions saved to risk_predictions table.")
print("\n(This is exactly Section 3's save_predictions(conn, oulad_risk, ...) call above.)")

Simulating: user selects option 18 (Train Risk Model), then enters '2' for OULAD



Training XGBoost classifier...

  Precision (at-risk): 0.887
  Recall   (at-risk): 0.783
  F1 (at-risk):       0.832
  ROC-AUC:            0.921

Model saved to: ../models/risk_model_oulad_v1.joblib
Predictions saved to risk_predictions table.

(This is exactly Section 3's save_predictions(conn, oulad_risk, ...) call above.)


In [11]:
print("=" * 60)
print("Simulating: user selects option 19 (Show Risk Predictions)")
print("=" * 60)
print()

preds = get_latest_predictions(conn, threshold=0.5)
print("At-risk students (risk_score >= 0.5):")
print(preds[["student_ref", "source_type", "risk_score", "predicted_label",
             "model_version", "scored_at"]].head(15).to_string(index=False))
print(f"\n... {len(preds):,} total at-risk students across both populations")

Simulating: user selects option 19 (Show Risk Predictions)



At-risk students (risk_score >= 0.5):
student_ref source_type  risk_score predicted_label    model_version           scored_at
      11685       oulad      1.0000         at_risk xgboost_oulad_v1 2026-08-12 19:36:07
      12503       oulad      1.0000         at_risk xgboost_oulad_v1 2026-08-12 19:36:07
      14584       oulad      1.0000         at_risk xgboost_oulad_v1 2026-08-12 19:36:07
      19695       oulad      1.0000         at_risk xgboost_oulad_v1 2026-08-12 19:36:07
      20226       oulad      1.0000         at_risk xgboost_oulad_v1 2026-08-12 19:36:07
      22446       oulad      1.0000         at_risk xgboost_oulad_v1 2026-08-12 19:36:08
      25788       oulad      1.0000         at_risk xgboost_oulad_v1 2026-08-12 19:36:08
       7493       oulad      1.0000         at_risk xgboost_oulad_v1 2026-08-12 19:36:07
       9622       oulad      1.0000         at_risk xgboost_oulad_v1 2026-08-12 19:36:07
      10014       oulad      0.9999         at_risk xgboost_oulad_v1 202

**What's genuinely simulated vs. what's real here:** the *data* (predictions,
counts, metrics) is 100% real — it's the same models and risk scores from every
notebook before this one, just read back through the actual `risk_queries.py`
functions against a real (if temporary) SQL database. What's simulated is only
the *terminal interaction* — the `input()` prompts and the `while True` menu
loop — since neither can run inside an executed notebook cell.

## 5. Summary — the complete Sprint 3 pipeline

| Module | Notebook | Status |
|---|---|---|
| EDA (UCI) | `eda_uci_student_performance.ipynb` | ✅ done |
| EDA (OULAD) | `eda_oulad.ipynb` | ✅ done |
| Feature engineering (UCI) | `feature_engineering_uci.ipynb` | ✅ done |
| Feature engineering (OULAD) | `feature_engineering_oulad.ipynb` | ✅ done |
| Model training + risk scores (UCI) | `model_training_uci.ipynb` | ✅ done |
| Model training + risk scores (OULAD) | `model_training_oulad.ipynb` | ✅ done |
| Clustering / performance personas | `clustering_performance_personas.ipynb` | ✅ done |
| Risk charts (Streamlit-ready) | `risk_charts.ipynb` | ✅ done |
| GRU sequence model (OULAD) | `model_gru_oulad.ipynb` | ✅ done |
| DB integration + CLI wiring | **this notebook** | ✅ done (SQLite stand-in) |

**What's left to do outside a notebook, in your actual project:**
1. Swap this notebook's SQLite connection for `db_manager.get_connection()`
   against your real MySQL instance — the query/insert *logic* needs no changes,
   only the connection object and placeholder style (`?` → `%s`).
2. Copy the `EngagementGRU` class into `src/risk/model_gru.py` so
   `torch.load()` can find it when reloading the checkpoint outside this notebook.
3. Wire the Streamlit app itself (Week 3 Day 10) — `risk_charts.ipynb`'s four
   functions are ready to drop into `st.pyplot(fig)` calls; `risk_queries.py`'s
   `get_latest_predictions()` is what the dashboard's student table would query.
4. `score_reflection_text()` (Hugging Face sentiment scoring, Week 4 Day 5-6)
   is genuinely a *different* sprint phase (NLP, not the ML/DL risk model) —
   worth its own notebook when you get there rather than folding it in here.